In [1]:
#!pip install osmnx folium geopy pandas --upgrade

In [ ]:
# Importar las librerías necesarias
import osmnx as ox
import networkx as nx
from folium.plugins import PolyLineTextPath
import pandas as pd
import heapq
from geopy.distance import geodesic
import math



In [3]:
#  FUNCIONES TOMTOM 
import os
import requests
import folium

TOMTOM_API_KEY = os.getenv("TOMTOM_API_KEY3")

if not TOMTOM_API_KEY:
    raise ValueError("ERROR: La variable de entorno TOMTOM_API_KEY no está configurada.")

# -------- Routing --------
def tomtom_route_eta(lat_o, lon_o, lat_d, lon_d):
    """
    Llama a la Routing API de TomTom y devuelve:
      - distance_m: distancia total en metros
      - travel_s: tiempo total de viaje (segundos, con condiciones actuales)
      - delay_s: retraso por tráfico (segundos, si TomTom lo reporta)
    """
    url = f"https://api.tomtom.com/routing/1/calculateRoute/{lat_o},{lon_o}:{lat_d},{lon_d}/json"

    params = {
        "key": TOMTOM_API_KEY,
        "routeType": "fastest",
        "traffic": "true",   # considerar tráfico
        "departAt": "now"    # condiciones actuales
    }

    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()

    summary = data["routes"][0]["summary"]

    distance_m = summary.get("lengthInMeters")
    travel_s   = summary.get("travelTimeInSeconds")
    delay_s    = summary.get("trafficDelayInSeconds", 0)

    return distance_m, travel_s, delay_s


def tomtom_from_graph_nodes(G, start_node, end_node):
    """
    Wrapper que toma nodos del grafo OSMnx y llama a tomtom_route_eta.
    """
    lat_o, lon_o = G.nodes[start_node]["y"], G.nodes[start_node]["x"]
    lat_d, lon_d = G.nodes[end_node]["y"], G.nodes[end_node]["x"]

    return tomtom_route_eta(lat_o, lon_o, lat_d, lon_d)


# -------- Traffic Flow  --------
def tomtom_flow_segment(lat, lon, zoom=10):
    """
    Llama a la Traffic Flow 'Flow Segment Data' API de TomTom, estilo 'absolute'.
    Devuelve el objeto flowSegmentData completo.
    Docs:
    https://developer.tomtom.com/traffic-api/documentation/tomtom-maps/traffic-flow/flow-segment-data
    """
    url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/{zoom}/json"
    params = {
        "key": TOMTOM_API_KEY,
        "point": f"{lat},{lon}",
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()
    return data["flowSegmentData"]

import time, pickle
from pathlib import Path
import requests

CACHE_DIR = Path("cache_tomtom")
CACHE_DIR.mkdir(exist_ok=True)
CACHE_TTL = 60 * 5  # 5 minutos

_mem_cache = {}

def _cache_key(lat, lon, zoom, decimals=4):
    return (zoom, round(lat, decimals), round(lon, decimals))

def tomtom_flow_segment_cached(lat, lon, zoom=10, timeout=5):
    key = _cache_key(lat, lon, zoom)
    now = time.time()

    # 1) cache RAM
    if key in _mem_cache:
        ts, data = _mem_cache[key]
        if now - ts < CACHE_TTL:
            return data

    # 2) cache disco
    f = CACHE_DIR / f"{key[0]}_{key[1]}_{key[2]}.pkl"
    if f.exists():
        ts, data = pickle.loads(f.read_bytes())
        if now - ts < CACHE_TTL:
            _mem_cache[key] = (ts, data)
            return data

    # 3) llamada real
    url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/{zoom}/json"
    params = {"key": TOMTOM_API_KEY, "point": f"{lat},{lon}"}
    resp = requests.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    data = resp.json()["flowSegmentData"]

    _mem_cache[key] = (now, data)
    f.write_bytes(pickle.dumps((now, data)))
    return data




def traffic_color_from_ratio(ratio, road_closure=False):
    """
    Convierte el ratio currentSpeed/freeFlowSpeed en un color tipo tráfico.
    """
    if road_closure:
        return "gray"   # calle cerrada

    if ratio is None:
        return "gray"    # sin datos 

    if ratio >= 0.9:
        return "green"   # fluido
    elif ratio >= 0.7:
        return "yellow"  # ligero
    elif ratio >= 0.4:
        return "orange"  # congestionado
    else:
        return "red"     # muy congestionado


def build_traffic_map_from_segments(segments, center_lat, center_lon, save_path=None):
    """
    Construye un mapa Folium con segmentos coloreados según el tráfico.
    segments: lista de dicts con claves:
      - 'coords': lista [(lat, lon), ...]
      - 'color': string ('green', 'yellow', 'orange', 'red', 'black', 'gray')
    """
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

    for seg in segments:
        coords = seg.get("coords", [])
        color = seg.get("color", "blue")
        if coords:
            folium.PolyLine(coords, color=color, weight=6).add_to(m)

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        m.save(save_path)

    return m



In [4]:
# CLIMA 
import time, pickle, requests
from pathlib import Path

WEATHER_CACHE_DIR = Path("cache_weather")
WEATHER_CACHE_DIR.mkdir(exist_ok=True)
WEATHER_TTL = 60 * 10  # 10 minutos
_weather_mem = {}

def _weather_key(lat, lon, decimals=3):
    return (round(lat, decimals), round(lon, decimals))

def weather_code_to_text(code):
    # Mapeo simple (puedes ampliar)
    m = {
        0:"Despejado", 1:"Mayormente despejado", 2:"Parcial nublado", 3:"Nublado",
        45:"Niebla", 48:"Niebla",
        51:"Llovizna", 53:"Llovizna", 55:"Llovizna",
        61:"Lluvia", 63:"Lluvia", 65:"Lluvia fuerte",
        80:"Chubascos", 81:"Chubascos", 82:"Chubascos fuertes",
        95:"Tormenta"
    }
    try:
        return m.get(int(code), f"Código {code}")
    except:
        return f"Código {code}"

def fetch_open_meteo_current(lat, lon, timeout=5):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": lat,
        "longitude": lon,
        "current_weather": "true",
        "timezone": "auto"
    }
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    j = r.json()

    if "current_weather" in j and j["current_weather"]:
        cw = j["current_weather"]
        return {
            "time": cw.get("time"),
            "temp": cw.get("temperature"),
            "wind": cw.get("windspeed"),
            "code": cw.get("weathercode"),
            "src": "open-meteo"
        }

    # Fallback por si el formato cambia
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,wind_speed_10m,weather_code",
        "timezone": "auto"
    }
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    j = r.json()
    c = j.get("current", {}) or {}

    return {
        "time": c.get("time"),
        "temp": c.get("temperature_2m"),
        "wind": c.get("wind_speed_10m"),
        "code": c.get("weather_code"),
        "src": "open-meteo"
    }

def get_weather_cached(lat, lon, timeout=5):
    key = _weather_key(lat, lon)
    now = time.time()

    # 1) RAM
    if key in _weather_mem:
        ts, data = _weather_mem[key]
        if now - ts < WEATHER_TTL:
            return data

    # 2) Disco
    f = WEATHER_CACHE_DIR / f"{key[0]}_{key[1]}.pkl"
    if f.exists():
        ts, data = pickle.loads(f.read_bytes())
        if now - ts < WEATHER_TTL:
            _weather_mem[key] = (ts, data)
            return data

    # 3) Llamada real
    data = fetch_open_meteo_current(lat, lon, timeout=timeout)
    _weather_mem[key] = (now, data)
    f.write_bytes(pickle.dumps((now, data)))
    return data

def agregar_clima_a_mapa(m, lat, lon, titulo="Clima (actual)"):
    try:
        w = get_weather_cached(lat, lon)
        desc = weather_code_to_text(w.get("code"))
        temp = w.get("temp")
        wind = w.get("wind")
        tstamp = w.get("time")

        html = f"""
        <div style="
            position: fixed;
            top: 50px; right: 50px;
            z-index: 9999;
            background-color: white;
            padding: 10px;
            border: 2px solid black;
            border-radius: 6px;
            font-size: 14px;
            max-width: 240px;
        ">
          <h4 style="margin:0 0 6px 0;">{titulo}</h4>
          <div><b>Temp:</b> {temp if temp is not None else "?"} °C</div>
          <div><b>Viento:</b> {wind if wind is not None else "?"} km/h</div>
          <div><b>Estado:</b> {desc}</div>
          <div style="font-size:11px; color:#555; margin-top:6px;">
            {tstamp if tstamp else ""} ({w.get("src","")})
          </div>
        </div>
        """
    except Exception:
        html = """
        <div style="
            position: fixed;
            top: 50px; right: 50px;
            z-index: 9999;
            background-color: white;
            padding: 10px;
            border: 2px solid black;
            border-radius: 6px;
            font-size: 14px;
            max-width: 240px;
        ">
          <h4 style="margin:0 0 6px 0;">Clima</h4>
          <div>Clima no disponible</div>
        </div>
        """

    m.get_root().html.add_child(folium.Element(html))
    return m


In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_traffic_for_edges_parallel(G, edges, zoom=10, max_workers=12):
    out = []

    def _one(u, v, k):
        return (u, v, k, get_traffic_for_edge(G, u, v, key=k, zoom=zoom))

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(_one, u, v, k) for (u, v, k) in edges]
        for fut in as_completed(futs):
            try:
                out.append(fut.result())  # (u,v,k,seg)
            except Exception:
                pass
    return out


In [6]:
def get_edge_coords_latlon(G, u, v, key=0):
    """
    Regresa la lista de coordenadas (lat, lon) de la arista (u, v, key).
    Si existe 'geometry', la usa; si no, usa los nodos u y v.
    """
    data = G.edges[u, v, key]

    # Caso 1: la arista tiene geometría (LineString)
    if "geometry" in data and data["geometry"] is not None:
        line = data["geometry"]
        # geometry.coords viene como (lon, lat), la invertimos a (lat, lon)
        return [(lat, lon) for (lon, lat) in line.coords]

    # Caso 2: sin geometría, solo un segmento recto entre u y v
    lat_u, lon_u = G.nodes[u]["y"], G.nodes[u]["x"]
    lat_v, lon_v = G.nodes[v]["y"], G.nodes[v]["x"]
    return [(lat_u, lon_u), (lat_v, lon_v)]


def get_traffic_for_edge(G, u, v, key=0, zoom=10):
    """
    Calcula el tráfico para la arista (u, v, key) usando TomTom.
    - Toma el punto medio de la arista.
    - Llama a tomtom_flow_segment(lat, lon).
    - Calcula ratio y color con traffic_color_from_ratio.
    Regresa un dict tipo 'segment' listo para agregar a traffic_segments.
    """
    # Coordenadas reales de la arista
    coords = get_edge_coords_latlon(G, u, v, key=key)

    # Punto medio de la arista
    mid_lat, mid_lon = coords[len(coords) // 2]

    # Llamada a TomTom
    flow = tomtom_flow_segment_cached(mid_lat, mid_lon, zoom=zoom)

    if not flow:
        return {
            "coords": coords,
            "color": "gray",
            "currentSpeed": None,
            "freeFlowSpeed": None,
            "ratio": None,
            "roadClosure": False,
        }

    curr = flow.get("currentSpeed")
    free = flow.get("freeFlowSpeed")
    road_closure = flow.get("roadClosure", False)

    if curr and free and free > 0:
        ratio = curr / free
    else:
        ratio = None

    color = traffic_color_from_ratio(ratio, road_closure)

    segment = {
        "coords": coords,
        "color": color,
        "currentSpeed": curr,
        "freeFlowSpeed": free,
        "ratio": ratio,
        "roadClosure": road_closure,
    }

    return segment


In [7]:
def mapa_ruta_con_trafico(G, astar_edges, traffic_segments,
                          origen="origen", destino="destino",
                          output_dir="Resultados/Mapas_Combinados"):
    import os
    import folium

    if not astar_edges:
        print("[INFO] No hay aristas en la ruta, no se genera mapa.")
        return None

    start_node = astar_edges[0][0]
    center_lat = G.nodes[start_node]["y"]
    center_lon = G.nodes[start_node]["x"]

    m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

    # Ruta en azul
    coords_route = []
    for (u, v) in astar_edges:
        coords_route.append((G.nodes[u]["y"], G.nodes[u]["x"]))
    last_v = astar_edges[-1][1]
    coords_route.append((G.nodes[last_v]["y"], G.nodes[last_v]["x"]))

    folium.PolyLine(coords_route, color="blue", weight=4, opacity=0.8,
                    tooltip="Ruta base / con tráfico").add_to(m)

    # Segmentos de tráfico coloreados
    if traffic_segments:
        for seg in traffic_segments:
            coords = seg.get("coords", [])
            color = seg.get("color", "gray")
            if coords:
                folium.PolyLine(
                    coords,
                    color=color,
                    weight=1,       # más delgado
                    opacity=0.35,   # más transparente
                    tooltip=(
                        f"Vel. actual: {seg.get('currentSpeed')} km/h | "
                        f"Libre: {seg.get('freeFlowSpeed')} km/h | "
                        f"ratio: {round(seg.get('ratio', 0), 2) if seg.get('ratio') else 'N/A'}"
                    )
                ).add_to(m)



    os.makedirs(output_dir, exist_ok=True)

    def slugify(name):
        return str(name).replace(" ", "_").replace("/", "-")

    filename = f"mapa_ruta_con_trafico_{slugify(origen)}_{slugify(destino)}.html"
    save_path = os.path.join(output_dir, filename)

    m.save(save_path)
    print(f"[OK] Mapa combinado guardado en: {save_path}")

    return m


In [8]:
def clean_graph(G, network_type):
    max_speed = {'drive': 40, 'walk': 5}
    for u, v, k in G.edges(keys=True):
        length = G.edges[u, v, k]["length"]
        max_speed_value = max_speed[network_type]

        # Penalización por tipo de calle
        penalty = 1.5 if "highway" in G.edges[u, v, k] and G.edges[u, v, k]["highway"] in ["residential", "tertiary"] else 1.0

        G.edges[u, v, k]["maxspeed"] = max_speed_value
        G.edges[u, v, k]["weight"] = (length / max_speed_value) * penalty
    return G

In [9]:
# Función para inicializar estilos de aristas
def initialize_edge_styles(G):
    for edge in G.edges:
        G.edges[edge]["color"] = "#d36206"
        G.edges[edge]["alpha"] = 0.2
        G.edges[edge]["linewidth"] = 0.5

In [10]:
# Solicitar las coordenadas de inicio y destino
def get_coordinates():
    while True:
        try:
            coords = input("Ingrese las coordenadas (latitud, longitud): ").strip()
            lat, lon = map(float, coords.split(","))
            return (lat, lon)
        except ValueError:
            print("Entrada inválida. Por favor, ingrese las coordenadas en el formato 'lat,lon'.")

In [11]:
# Solicitar el tipo de transporte
def get_transport_type():
    while True:
        print("Ingrese el tipo de viaje que realizará:")
        print("  1. Auto")
        print("  2. Caminando")
        try:
            option = int(input("Seleccione una opción (1 o 2): ").strip())
            if option in [1, 2]:
                return 'drive' if option == 1 else 'walk'
            else:
                print("Opción inválida. Por favor seleccione 1 o 2.")
        except ValueError:
            print("Entrada inválida. Por favor ingrese un número (1 o 2).")

In [12]:
# Algoritmo de Dijkstra
def dijkstra(G, orig, dest):
    for node in G.nodes:
        G.nodes[node]["visited"] = False
        G.nodes[node]["distance"] = float("inf")
        G.nodes[node]["previous"] = None
    G.nodes[orig]["distance"] = 0
    pq = [(0, orig)]
    while pq:
        _, node = heapq.heappop(pq)
        if node == dest:
            break
        if G.nodes[node]["visited"]:
            continue
        G.nodes[node]["visited"] = True
        for u, v, k in G.out_edges(node, keys=True):
            weight = G.edges[u, v, k]["weight"]
            if G.nodes[v]["distance"] > G.nodes[node]["distance"] + weight:
                G.nodes[v]["distance"] = G.nodes[node]["distance"] + weight
                G.nodes[v]["previous"] = node
                heapq.heappush(pq, (G.nodes[v]["distance"], v))

In [13]:
# Algoritmo A*
def a_star(G, orig, dest):
    def heuristic(node1, node2):
        x1, y1 = G.nodes[node1]["x"], G.nodes[node1]["y"]
        x2, y2 = G.nodes[node2]["x"], G.nodes[node2]["y"]
        return geodesic((y1, x1), (y2, x2)).meters  # Distancia geodésica en metros

    for node in G.nodes:
        G.nodes[node]["g_score"] = float("inf")
        G.nodes[node]["f_score"] = float("inf")
        G.nodes[node]["previous"] = None

    G.nodes[orig]["g_score"] = 0
    G.nodes[orig]["f_score"] = heuristic(orig, dest)
    pq = [(G.nodes[orig]["f_score"], orig)]

    while pq:
        _, node = heapq.heappop(pq)
        if node == dest:
            break
        for u, v, k in G.out_edges(node, keys=True):
            tentative_g_score = G.nodes[node]["g_score"] + G.edges[u, v, k]["weight"]
            if tentative_g_score < G.nodes[v]["g_score"]:
                G.nodes[v]["g_score"] = tentative_g_score
                G.nodes[v]["f_score"] = tentative_g_score + heuristic(v, dest)
                G.nodes[v]["previous"] = node
                heapq.heappush(pq, (G.nodes[v]["f_score"], v))


In [14]:
def get_path_edges_from_previous(G, start_node, end_node):
    path_nodes = []
    curr = end_node
    while curr != start_node:
        prev = G.nodes[curr].get("previous", None)
        if prev is None:
            return None
        path_nodes.append((prev, curr))
        curr = prev
    path_nodes.reverse()
    return path_nodes


def get_path_edges_networkx(G, start_node, end_node):
    real_nodes = nx.shortest_path(G, start_node, end_node, weight="weight")
    real_edges = []
    for u, v in zip(real_nodes[:-1], real_nodes[1:]):
        real_edges.append((u, v))
    return real_edges


def compare_paths(pred_edges, real_edges):
    if pred_edges is None or real_edges is None:
        return {
            "tp": 0, "fp": 0, "fn": 0,
            "precision": 0.0, "recall": 0.0, "f1": 0.0
        }

    # Se normalizan los arcos 
    norm_pred = {tuple(sorted(e)) for e in pred_edges}
    norm_real = {tuple(sorted(e)) for e in real_edges}

    tp = len(norm_pred & norm_real)
    fp = len(norm_pred - norm_real)
    fn = len(norm_real - norm_pred)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}


def evaluate_pair(G, start_node, end_node, network_type="drive"):
    """
    Evalúa un par (origen, destino) en el grafo:
      - Dijkstra base (sin tráfico)
      - A* base (sin tráfico)
      - Penaliza pesos según tráfico en la ruta A* base (TomTom Traffic Flow)
      - Dijkstra con tráfico (pesos penalizados)
      - A* con tráfico (pesos penalizados)
      - Métricas de tiempo, distancia, precisión/recall para todas las variantes
      - TomTom Routing para ETA global
      - TomTom Traffic Flow sobre la ruta final A* con tráfico para colorear el mapa
    """
    results = {}

    #  1) DIJKSTRA BASE (sin tráfico) 
    t0 = time.perf_counter()
    dijkstra(G, start_node, end_node)
    t1 = time.perf_counter()
    dijkstra_time_base = (t1 - t0) * 1000  # ms

    dijkstra_edges_base = get_path_edges_from_previous(G, start_node, end_node)
    if dijkstra_edges_base:
        dijkstra_dist_base = sum(G.edges[u, v, 0]["length"] for (u, v) in dijkstra_edges_base)
    else:
        dijkstra_dist_base = math.inf

    #  2) A* BASE (sin tráfico) 
    t0 = time.perf_counter()
    a_star(G, start_node, end_node)
    t1 = time.perf_counter()
    astar_time_base = (t1 - t0) * 1000  # ms

    astar_edges_base = get_path_edges_from_previous(G, start_node, end_node)
    if astar_edges_base:
        astar_dist_base = sum(G.edges[u, v, 0]["length"] for (u, v) in astar_edges_base)
    else:
        astar_dist_base = math.inf

    #  3) Ruta "real" de referencia (networkx) 
    real_edges = get_path_edges_networkx(G, start_node, end_node)

    d_base_metrics   = compare_paths(dijkstra_edges_base, real_edges)
    a_base_metrics   = compare_paths(astar_edges_base, real_edges)

    #  4) Penalizar pesos según tráfico en ruta A* base 
    max_factor = 3.0  # tope de penalización (3x el peso original)


 
    # 4) AJUSTE DE PESOS SEGÚN TRÁFICO TOMTOM (POR ARISTA)
    traffic_segments = []
    penalized_edges_info = []
    
    if astar_edges_base:
        # 1) armar lista (u,v,k) a consultar
        edges_to_query = []
        for (u, v) in astar_edges_base:
            try:
                for k in G[u][v]:
                    edges_to_query.append((u, v, k))
            except Exception:
                pass
    
        # 2) pedir tráfico en paralelo
        traffic_results = get_traffic_for_edges_parallel(G, edges_to_query, zoom=10, max_workers=12)
    
        # 3) aplicar penalización usando los resultados 
        for (u, v, k, seg) in traffic_results:
            try:
                curr = seg["currentSpeed"]
                free = seg["freeFlowSpeed"]
                road_closure = seg["roadClosure"]
    
                factor = 1.0
                if road_closure:
                    factor = max_factor
                elif curr and free and curr > 0:
                    ratio = curr / free
                    if ratio < 1.0:
                        factor = min(1.0 / ratio, max_factor)
    
                if "weight" in G[u][v][k]:
                    old_weight = G[u][v][k]["weight"]
                    new_weight = old_weight * factor
                    G[u][v][k]["weight"] = new_weight
    
                    penalized_edges_info.append({
                        "u": u, "v": v, "key": k,
                        "old_weight": old_weight,
                        "new_weight": new_weight,
                        "factor": factor,
                        "currentSpeed": curr,
                        "freeFlowSpeed": free,
                        "ratio": seg.get("ratio"),
                        "roadClosure": road_closure,
                    })
    
                traffic_segments.append(seg)
    
            except Exception as e:
                print(f"[WARN] No se pudo ajustar tráfico en {u}->{v} (key={k}): {e}")
                continue
    
    results["traffic_segments"] = traffic_segments
    results["penalized_edges"] = penalized_edges_info

 
    #  5) DIJKSTRA CON TRÁFICO (pesos penalizados) 
    t0 = time.perf_counter()
    dijkstra(G, start_node, end_node)
    t1 = time.perf_counter()
    dijkstra_time_traffic = (t1 - t0) * 1000  # ms

    dijkstra_edges_traffic = get_path_edges_from_previous(G, start_node, end_node)
    if dijkstra_edges_traffic:
        dijkstra_dist_traffic = sum(G.edges[u, v, 0]["length"] for (u, v) in dijkstra_edges_traffic)
    else:
        dijkstra_dist_traffic = math.inf

    d_traffic_metrics = compare_paths(dijkstra_edges_traffic, real_edges)

    #  6) A* CON TRÁFICO (pesos penalizados) 
    t0 = time.perf_counter()
    a_star(G, start_node, end_node)
    t1 = time.perf_counter()
    astar_time_traffic = (t1 - t0) * 1000  # ms

    astar_edges_traffic = get_path_edges_from_previous(G, start_node, end_node)
    if astar_edges_traffic:
        astar_dist_traffic = sum(G.edges[u, v, 0]["length"] for (u, v) in astar_edges_traffic)
    else:
        astar_dist_traffic = math.inf

    a_traffic_metrics = compare_paths(astar_edges_traffic, real_edges)
    results["astar_edges_base"] = astar_edges_base
    results["astar_edges_traffic"] = astar_edges_traffic


    #  7) Speedups y eficiencia 
    speedup_A_base      = dijkstra_time_base    / astar_time_base    if astar_time_base    > 0 else 0.0
    eficiencia_A_base   = speedup_A_base / 1.0

    speedup_A_traffic   = dijkstra_time_base    / astar_time_traffic if astar_time_traffic > 0 else 0.0
    eficiencia_A_traf   = speedup_A_traffic / 1.0

    speedup_D_traffic   = dijkstra_time_base    / dijkstra_time_traffic if dijkstra_time_traffic > 0 else 0.0
    eficiencia_D_traf   = speedup_D_traffic / 1.0

    #  8) TomTom ROUTING (ETA global tramo start->end) 
    try:
        tt_dist_m, tt_time_s, tt_delay_s = tomtom_from_graph_nodes(G, start_node, end_node)
    except Exception as e:
        print(f"[WARN] Error al llamar a TomTom Routing para {start_node} -> {end_node}: {e}")
        tt_dist_m, tt_time_s, tt_delay_s = None, None, None

    #  9) TomTom TRAFFIC FLOW sobre la ruta FINAL A* (con tráfico) 
    traffic_segments = []
    ratios = []

    if astar_edges_traffic:
        for (u, v) in astar_edges_traffic:
            try:
                lat_u, lon_u = G.nodes[u]["y"], G.nodes[u]["x"]
                flow = tomtom_flow_segment_cached(lat_u, lon_u, zoom=10)


                curr = flow.get("currentSpeed")
                free = flow.get("freeFlowSpeed")
                road_closure = flow.get("roadClosure", False)
                coords_raw = flow.get("coordinates", {}).get("coordinate", [])

                if free and free > 0:
                    ratio = curr / free
                else:
                    ratio = None

                color = traffic_color_from_ratio(ratio, road_closure)
                coords = [(c["latitude"], c["longitude"]) for c in coords_raw]

                traffic_segments.append({
                    "coords": coords,
                    "currentSpeed": curr,
                    "freeFlowSpeed": free,
                    "ratio": ratio,
                    "roadClosure": road_closure,
                    "color": color,
                })

                if ratio is not None:
                    ratios.append(ratio)

            except Exception as e:
                print(f"[WARN] Error en Traffic Flow para arista {u}->{v}: {e}")
                continue

    if ratios:
        avg_ratio = sum(ratios) / len(ratios)
        congested_fraction = sum(1 for r in ratios if r < 0.7) / len(ratios)
    else:
        avg_ratio = None
        congested_fraction = None

    #  10) Guardar todo en results 

    # Dijkstra base
    results["dijkstra_time_ms"] = dijkstra_time_base
    results["dijkstra_dist_m"] = dijkstra_dist_base
    results["dijkstra_precision"] = d_base_metrics["precision"]
    results["dijkstra_recall"] = d_base_metrics["recall"]
    results["dijkstra_f1"] = d_base_metrics["f1"]

    # Dijkstra con tráfico
    results["dijkstra_traffic_time_ms"] = dijkstra_time_traffic
    results["dijkstra_traffic_dist_m"] = dijkstra_dist_traffic
    results["dijkstra_traffic_precision"] = d_traffic_metrics["precision"]
    results["dijkstra_traffic_recall"] = d_traffic_metrics["recall"]
    results["dijkstra_traffic_f1"] = d_traffic_metrics["f1"]
    results["speedup_Dijkstra_traffic_vs_base"] = speedup_D_traffic
    results["eficiencia_Dijkstra_traffic"] = eficiencia_D_traf

    # A* base
    results["astar_time_ms"] = astar_time_base
    results["astar_dist_m"] = astar_dist_base
    results["astar_precision"] = a_base_metrics["precision"]
    results["astar_recall"] = a_base_metrics["recall"]
    results["astar_f1"] = a_base_metrics["f1"]
    results["speedup_A*_vs_Dijkstra"] = speedup_A_base
    results["eficiencia_A*_base"] = eficiencia_A_base

    # A* con tráfico
    results["astar_traffic_time_ms"] = astar_time_traffic
    results["astar_traffic_dist_m"] = astar_dist_traffic
    results["astar_traffic_precision"] = a_traffic_metrics["precision"]
    results["astar_traffic_recall"] = a_traffic_metrics["recall"]
    results["astar_traffic_f1"] = a_traffic_metrics["f1"]
    results["speedup_A*_traffic_vs_Dijkstra_base"] = speedup_A_traffic
    results["eficiencia_A*_traffic"] = eficiencia_A_traf

    # TomTom Routing
    results["tomtom_dist_m"] = tt_dist_m
    results["tomtom_time_s"] = tt_time_s
    results["tomtom_delay_s"] = tt_delay_s

    # Métricas de Traffic Flow (ruta A* final)
    results["traffic_avg_speed_ratio"] = avg_ratio
    results["traffic_congested_fraction"] = congested_fraction
    results["traffic_segments_flow"] = traffic_segments 

    
    results["eficiencia"] = results["eficiencia_A*_base"]

    
    return results



def eval_task(args):
    """
    Tarea que se puede correr en paralelo.
    Se hace una copia del grafo para no pisar los 'previous' de otros hilos.
    """
    G, start_node, end_node, start_name, end_name, network_type = args
    G_local = G.copy()  # <- cada proceso/hilo trabaja con su copia

    r = evaluate_pair(G_local, start_node, end_node, network_type=network_type)
    r["origen"] = start_name
    r["destino"] = end_name
    return r


from concurrent.futures import ProcessPoolExecutor

def evaluate_pairs_parallel(G, nodes, point_names, network_type="drive", max_workers=4):
    """
    Versión paralela basada en ProcessPoolExecutor (esta puede ser
    sobrescrita en la celda 17 con la versión que usa ThreadPool en Windows).
    """
    tasks = []
    for i in range(len(point_names) - 1):
        start_name = point_names[i]
        end_name = point_names[i + 1]
        start_node = nodes[start_name]
        end_node = nodes[end_name]
        tasks.append((G, start_node, end_node, start_name, end_name, network_type))

    results = []
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        for r in ex.map(eval_task, tasks):
            results.append(r)

    return results



In [15]:
# Mostrar resultados en tabla
def show_results_in_table(total_time, total_distance):
    """
    Muestra los resultados finales en una tabla.
    """
    data = {
        "Algoritmo": ["Dijkstra", "A*"],
        "Distancia Total (km)": [
            total_distance["Dijkstra"] / 1000,  # Convertir de metros a kilómetros
            total_distance["A*"] / 1000
        ],
        "Tiempo Total (minutos)": [
            total_time["Dijkstra"],
            total_time["A*"]
        ]
    }

    # Crear el DataFrame y mostrarlo
    df_results = pd.DataFrame(data)
    display(df_results)


In [16]:
# Obtener las coordenadas de la ruta más corta
def get_route_coordinates(G, orig, dest):
    path = []
    curr = dest
    while curr != orig:
        prev = G.nodes[curr]["previous"]
        if prev is None:
            print("No se pudo encontrar un camino desde el origen al destino.")
            return None
        path.append((G.nodes[curr]["y"], G.nodes[curr]["x"]))
        curr = prev
    path.append((G.nodes[orig]["y"], G.nodes[orig]["x"]))
    path.reverse()
    return path

In [17]:
# Visualizar la ruta con Folium
def plot_route_with_folium(G, orig, dest, route_coordinates):
    folium_map = folium.Map(
        location=[G.nodes[orig]['y'], G.nodes[orig]['x']],
        zoom_start=14,
        tiles='OpenStreetMap'
    )
    folium.Marker(
        location=[G.nodes[orig]['y'], G.nodes[orig]['x']],
        popup="Inicio",
        icon=folium.Icon(color="green", icon="play"),
    ).add_to(folium_map)
    folium.Marker(
        location=[G.nodes[dest]['y'], G.nodes[dest]['x']],
        popup="Destino",
        icon=folium.Icon(color="red", icon="stop"),
    ).add_to(folium_map)
    folium.PolyLine(
        route_coordinates,
        color="blue",
        weight=5,
        opacity=1,
        tooltip="Ruta más corta"
    ).add_to(folium_map)
    return folium_map

In [18]:
# Coordenadas guardadas
saved_coordinates = {
    "UPIIT": (19.323118, -98.233548),
    "Parque de Zacatelco": (19.215691, -98.240524),
    "Soriana Ocotlan": (19.318605, -98.220713),
    "Zoologico del Altiplano": (19.338439, -98.199046),
    "Recinto Ferial": (19.32500854887219, -98.24458295157108),
    "Escalinatas": (19.3163317057408, -98.2432146346621),
    "Jardín botánico": (19.328860233644402, -98.2186988643214),
    "Museo Nacional del Títere": (19.31306644075224, -97.92325948735258),
    "Museo Taurino de Huamantla": (19.315214643452563, -97.92088832789638),
    "Parque de Apizaco": (19.41583668098076, -98.1404040514315),
    "Parque de Tlaxco": (19.614479362946597, -98.11878748334519),
    "Centro Turístico Zacatelco": (19.201482353314457, -98.250818177072),
    "Val'Quirico": (19.19134212101139, -98.28932538116192),
    "Cacaxtla": (19.244591586170426, -98.33845364678213),
}

# Función para obtener coordenadas (seleccionar de lista o ingresar manualmente)
def get_coordinates_with_saved_options():
    print("\n¿Desea seleccionar un punto de la lista predefinida o ingresar coordenadas manualmente?")
    print("  1. Seleccionar de la lista")
    print("  2. Ingresar manualmente")
    while True:
        try:
            option = int(input("Seleccione una opción (1 o 2): ").strip())
            if option == 1:
                print("\nPuntos disponibles:")
                for idx, (name, coord) in enumerate(saved_coordinates.items(), start=1):
                    print(f"  {idx}. {name} - Coordenadas: {coord}")
                while True:
                    try:
                        choice = int(input("Seleccione un punto por su número: ").strip())
                        if 1 <= choice <= len(saved_coordinates):
                            selected_name = list(saved_coordinates.keys())[choice - 1]
                            print(f"Ha seleccionado: {selected_name} - Coordenadas: {saved_coordinates[selected_name]}")
                            coord = saved_coordinates[selected_name]
                            return coord, selected_name

                        else:
                            print("Por favor, seleccione un número válido de la lista.")
                    except ValueError:
                        print("Entrada inválida. Por favor, ingrese un número.")
            elif option == 2:
                coord = get_coordinates()  # Llama a la función manual existente
                return coord, "Coordenadas manuales"

            else:
                print("Por favor, seleccione una opción válida (1 o 2).")
        except ValueError:
            print("Entrada inválida. Por favor, ingrese un número.")


In [19]:
# Función para agregar puntos dinámicamente
def get_multiple_points():
    points = {}        # nombre del punto -> coordenadas
    point_labels = {}  # nombre del punto -> etiqueta (UPIIT, Soriana, etc.)
    point_counter = 0  # Contador para nombrar los puntos dinámicamente (A, B, C, etc.)

    while True:
        # Generar el nombre del punto (A, B, C, ...)
        point_name = chr(65 + point_counter)  # 65 es el código ASCII de 'A'
        print(f"\nIngrese las coordenadas para el punto {point_name}:")

        # Obtener coordenadas + etiqueta
        coord, label = get_coordinates_with_saved_options()
        points[point_name] = coord          # Guardar coordenadas
        point_labels[point_name] = label    # Guardar nombre de la ubicación

        # Preguntar si desea agregar otro punto
        while True:
            add_more = input(f"¿Desea agregar otro punto después de {point_name}? (s/n): ").strip().lower()
            if add_more in ["s", "n"]:
                break
            print("Por favor, ingrese 's' para sí o 'n' para no.")

        if add_more == "n":
            break  # Finalizar el bucle si el usuario no desea agregar más puntos

        point_counter += 1  # Incrementar el contador para el próximo punto

    return points, point_labels



In [20]:
import random

def generate_random_color():
    """
    Genera un color hexadecimal aleatorio.
    """
    return f"#{random.randint(0, 255):02x}{random.randint(0, 255):02x}{random.randint(0, 255):02x}"

def plot_full_route_with_folium(G, nodes, routes, algorithm_name, point_labels=None):

    """
    Genera un mapa para un algoritmo específico con puntos y rutas.
    """
    # Crear el mapa centrado en el primer punto
    first_point = list(nodes.values())[0]
    folium_map = folium.Map(
        location=[G.nodes[first_point]['y'], G.nodes[first_point]['x']],
        zoom_start=12,
        tiles='OpenStreetMap'
    )

    predefined_colors = [
        "blue", "red", "green", "purple", "orange", "pink",
        "darkred", "lightred", "beige", "darkblue", "darkgreen",
        "cadetblue", "darkpurple", "lightblue", "lightgreen",
        "gray", "black", "lightgray"
    ]
    color_map = {}

    legend_items = []
    for idx, (name, node) in enumerate(nodes.items()):
        # Elegir un color de la lista; si hay más puntos, se reciclan
        color = predefined_colors[idx % len(predefined_colors)]
        color_map[name] = color

        # Construir texto "Punto A: UPIIT"
        etiqueta = None
        if point_labels and name in point_labels:
            etiqueta = point_labels[name]

        popup_text = f"Punto {name}"
        if etiqueta:
            popup_text += f": {etiqueta}"

        # Usar Marker estándar 
        folium.Marker(
            location=[G.nodes[node]['y'], G.nodes[node]['x']],
            popup=popup_text,
            icon=folium.Icon(color=color, icon="info-sign"),
        ).add_to(folium_map)

        # En la leyenda se muestra el mismo texto
        legend_items.append(f"<span style='color:{color};'>{popup_text}</span>")



    # Añadir rutas al mapa
    for start_name, end_name, route_coordinates in routes:
        # 1) Dibujar la línea base de la ruta 
        folium.PolyLine(
            route_coordinates,
            color="blue",
            weight=5,
            opacity=1,
            tooltip=f"Ruta de {start_name} a {end_name} ({algorithm_name})"
        ).add_to(folium_map)

        # 2) Se añade una "flecha" en el último punto de la ruta
        if len(route_coordinates) >= 2:
            # Se toman los dos últimos puntos para calcular la orientación
            (lat_prev, lon_prev) = route_coordinates[-2]
            (lat_end, lon_end)  = route_coordinates[-1]

            # Calcular ángulo de la flecha (en grados)
            angle_rad = math.atan2(lat_end - lat_prev, lon_end - lon_prev)
            angle_deg = math.degrees(angle_rad)

            # Dibujar un triángulo que apunta en la dirección de la ruta
            folium.RegularPolygonMarker(
                location=(lat_end, lon_end),
                number_of_sides=3,       # triángulo
                radius=10,               # tamaño de la flecha
                rotation=angle_deg,      # orientación
                color="blue",
                fill=True,
                fill_color="blue",
                fill_opacity=0.9
            ).add_to(folium_map)



    # Añadir leyenda al mapa
    legend_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; background-color: white; padding: 10px; border: 2px solid black;">
    <h4>Leyenda</h4>
    """
    legend_html += "".join(f"<p style='margin:0'>{item}</p>" for item in legend_items)
    legend_html += "</div>"
    folium_map.get_root().html.add_child(folium.Element(legend_html))

    return folium_map



In [21]:

def agregar_trafico_a_mapa(m, traffic_segments):
    """
    Recibe un mapa Folium ya creado (con rutas Dijkstra/A*)
    y dibuja encima los segmentos de tráfico de TomTom con colores.

    - m: folium.Map (ya con rutas)
    - traffic_segments: lista de dicts como results["traffic_segments"]
    """
    if not traffic_segments:
        # Nada que pintar
        return m

    for seg in traffic_segments:
        coords = seg.get("coords", [])
        color = seg.get("color", "gray")
        if not coords:
            continue

        folium.PolyLine(
            coords,
            color=color,       # verde / amarillo / naranja / rojo / negro
            weight=9,          
            opacity=0.3,
            tooltip=(
                f"Vel. actual: {seg.get('currentSpeed')} km/h | "
                f"Libre: {seg.get('freeFlowSpeed')} km/h | "
                f"ratio: {round(seg.get('ratio', 0), 2) if seg.get('ratio') else 'N/A'}"
            )
        ).add_to(m)

    return m


In [22]:
import itertools
from geopy.distance import geodesic

def segmento_cerca_de_ruta(segment_coords, ruta_coords, max_m=150):
    """
    Regresa True si ALGÚN punto del segmento de tráfico está
    a menos de max_m metros de ALGÚN punto de la ruta.
    """
    for (lat1, lon1) in segment_coords:
        for (lat2, lon2) in ruta_coords:
            if geodesic((lat1, lon1), (lat2, lon2)).meters <= max_m:
                return True
    return False


def filtrar_segmentos_por_ruta(segments, rutas_coords, max_m=150):
    """
    - segments: lista de dicts con 'coords' (segmentos de tráfico).
    - rutas_coords: lista de listas de coordenadas de TODAS las rutas
      de un algoritmo (lo que devuelve plot_full_route_with_folium).
    Regresa SOLO los segmentos cuyo trazo está cerca de la ruta.
    """
    # Aplanar todos los puntos de las rutas del algoritmo
    all_route_points = list(itertools.chain.from_iterable(rutas_coords))
    filtrados = []

    for seg in segments:
        coords_seg = seg.get("coords", [])
        if not coords_seg:
            continue

        if segmento_cerca_de_ruta(coords_seg, all_route_points, max_m=max_m):
            filtrados.append(seg)

    return filtrados


In [23]:
import os
import pickle

GRAPH_PATH = "tlaxcala_drive.pkl" # Grafo ya existente para que no tarde mucho

def load_tlaxcala_graph(network_type="drive"):
    graph_path = f"tlaxcala_{network_type}.pkl"
    if os.path.exists(graph_path):
        print("Cargando grafo desde disco...")
        with open(graph_path, "rb") as f:
            G = pickle.load(f)
    else:
        print("Descargando el grafo de Tlaxcala...")
        tlaxcala_graph = ox.graph_from_place("Tlaxcala, México", network_type=network_type)
        G = clean_graph(tlaxcala_graph, network_type)
        initialize_edge_styles(G)
        with open(graph_path, "wb") as f:
            pickle.dump(G, f)
        print("Grafo guardado en", graph_path)
    return G


In [24]:
import os
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def eval_task_light(args):
    start_node, end_node, start_name, end_name, network_type = args

    G_local = load_tlaxcala_graph(network_type=network_type)
    r = evaluate_pair(G_local, start_node, end_node, network_type=network_type)
    r["origen"] = start_name
    r["destino"] = end_name
    return r

def evaluate_pairs_parallel(G, nodes, point_names, network_type="drive", max_workers=4):
    tasks = []
    for i in range(len(point_names) - 1):
        sname = point_names[i]
        ename = point_names[i + 1]
        snode = nodes[sname]
        enode = nodes[ename]
        tasks.append((snode, enode, sname, ename, network_type))

    results = []

    Executor = ThreadPoolExecutor if os.name == "nt" else ProcessPoolExecutor

    with Executor(max_workers=max_workers) as ex:
        for r in ex.map(eval_task_light, tasks):
            results.append(r)

    return results


In [ ]:
if __name__ == "__main__":
    
    try:
        print("Seleccione los puntos de la ruta:")
        print("Seleccione los puntos de la ruta:")
        points, point_labels = get_multiple_points()
        print("\nPuntos ingresados:")
        for name, coord in points.items():
            etiqueta = point_labels.get(name, "")
            if etiqueta:
                print(f"  {name}: {coord} -> {etiqueta}")
            else:
                print(f"  {name}: {coord}")

    
        # Tipo de transporte
        network_type = get_transport_type()
    
        # Cargar/descargar grafo
        G = load_tlaxcala_graph(network_type=network_type)
    
        # Obtener nodos más cercanos
        nodes = {}
        for name, coord in points.items():
            nodes[name] = ox.distance.nearest_nodes(G, coord[1], coord[0])
            print(f"Nodo más cercano al punto {name}: {nodes[name]}")
    
        point_names = list(points.keys())
    
        algorithms = {"Dijkstra": dijkstra, "A*": a_star}
        total_time = {"Dijkstra": 0, "A*": 0}
        total_distance = {"Dijkstra": 0, "A*": 0}
        routes_by_algorithm = {"Dijkstra": [], "A*": []}

        #  1) MÉTRICAS EN PARALELO 
        metrics_rows = evaluate_pairs_parallel(
            G,
            nodes,
            point_names,
            network_type=network_type,
            max_workers=4
        )
        for r in metrics_rows:
            edges_traf = r.get("astar_edges_traffic", [])
            segments = r.get("traffic_segments", [])
            origen = r.get("origen", "A")
            destino = r.get("destino", "B")

            print(f"Generando mapa para {origen} -> {destino}...")
            mapa_ruta_con_trafico(G, edges_traf, segments, origen, destino)

    
        #  2) RECORRIDO NORMAL (PARA MAPAS Y TOTALES) 
        for i in range(len(point_names) - 1):
            start_name = point_names[i]
            end_name = point_names[i + 1]
            print(f"\nCalculando la ruta de {start_name} a {end_name}...")
    
            start_node = nodes[start_name]
            end_node = nodes[end_name]
    
            for algorithm_name, algorithm in algorithms.items():
                print(f"Ejecutando el algoritmo {algorithm_name} de {start_name} a {end_name}...")
                try:
                    algorithm(G, start_node, end_node)
    
                    # reconstruir ruta
                    path = []
                    curr = end_node
                    while curr != start_node:
                        prev = G.nodes[curr]["previous"]
                        path.append((prev, curr))
                        curr = prev
                    path.reverse()
    
                    # distancia y tiempo estimado
                    try:
                        dist = sum(G.edges[u, v, 0]["length"] for u, v in path)
                        avg_speed = 40 if network_type == 'drive' else 5
                        time_est = (dist / 1000) / avg_speed * 60
    
                        total_distance[algorithm_name] += dist
                        total_time[algorithm_name] += time_est
    
                        print(f"{algorithm_name}:")
                        print(f"  Distancia estimada de {start_name} a {end_name}: {dist / 1000:.2f} km")
                        print(f"  Tiempo estimado de {start_name} a {end_name}: {time_est:.2f} minutos")
    
                        # guardar para folium
                        route_coordinates = get_route_coordinates(G, start_node, end_node)
                        routes_by_algorithm[algorithm_name].append(
                            (start_name, end_name, route_coordinates)
                        )
    
                    except Exception as e:
                        print(f"Error al calcular la distancia o tiempo para {algorithm_name}: {e}")
    
                except Exception as e:
                    print(f"Error durante la ejecución de {algorithm_name}: {e}")

    
        #  3) MOSTRAR MÉTRICAS 
        if metrics_rows:
            df_eval = pd.DataFrame(metrics_rows)
            cols = [
                "origen", "destino",
                "dijkstra_time_ms", "astar_time_ms",
                "speedup_A*_vs_Dijkstra", "eficiencia",
                "dijkstra_dist_m", "astar_dist_m",
                "dijkstra_precision", "dijkstra_recall", "dijkstra_f1",
                "astar_precision", "astar_recall", "astar_f1"
            ]
            display(df_eval[cols])
    
            print("\n== Resumen global de métricas ==")
            print("Tiempo medio Dijkstra (ms):", df_eval["dijkstra_time_ms"].mean())
            print("Tiempo medio A* (ms):", df_eval["astar_time_ms"].mean())
            print("Speedup medio A* vs Dijkstra:",
                  (df_eval["dijkstra_time_ms"] / df_eval["astar_time_ms"]).mean())
            print("F1 Dijkstra:", df_eval["dijkstra_f1"].mean())
            print("F1 A*:", df_eval["astar_f1"].mean())

            df_eval.to_csv("resultados_iniciales_tlaxcala.csv", index=False)

        #  4) TOTALES 
        print("\n--- Totales Finales ---")
        for algorithm_name in algorithms.keys():
            print(f"{algorithm_name}:")
            print(f"  Distancia total estimada: {total_distance[algorithm_name] / 1000:.2f} km")
            print(f"  Tiempo total estimado: {total_time[algorithm_name]:.2f} minutos")
    
        show_results_in_table(total_time, total_distance)

    
        #  5) MAPAS 
        for algorithm_name, routes in routes_by_algorithm.items():
        
            print(f"\nMostrando mapa para el algoritmo {algorithm_name}...")
        
            # 1. Crear el mapa base con la ruta del algoritmo
            folium_map = plot_full_route_with_folium(G, nodes, routes, algorithm_name, point_labels)
            # Coordenada “representativa” para clima: centro de los puntos seleccionados
            lats = [G.nodes[n]["y"] for n in nodes.values()]
            lons = [G.nodes[n]["x"] for n in nodes.values()]
            lat_c = sum(lats) / len(lats)
            lon_c = sum(lons) / len(lons)
            
            agregar_clima_a_mapa(folium_map, lat_c, lon_c)

        
            # 1.5. Juntar TODAS las coordenadas de ruta de este algoritmo
            all_route_coords = [rc for (_, _, rc) in routes]
        
            # 2. Añadir SOLO el tráfico de cada tramo correspondiente,
            #    filtrado para que esté cerca de la ruta azul.
            for (start_name, end_name, route_coordinates) in routes:
                for r in metrics_rows:
                    if r["origen"] == start_name and r["destino"] == end_name:
                        segments = r.get("traffic_segments", [])
        
                        # --- FILTRO CLAVE ---
                        segments_filtrados = filtrar_segmentos_por_ruta(
                            segments,
                            all_route_coords,
                            max_m=150   
                        )
        
                        agregar_trafico_a_mapa(folium_map, segments_filtrados)
                        break  # Ya encontramos el tramo, no seguir buscando
        
            # 3. Mostrar el mapa final
            display(folium_map)


    except Exception as e: print(f"Error general: {e}")

#  GENERACIÓN DE MAPAS DE TRÁFICO 

def generar_mapas_trafico(metrics_rows, output_dir="Resultados/Mapas_Trafico"):
    """
    Genera un mapa HTML por cada par origen–destino, usando los segmentos de tráfico
    guardados en results["traffic_segments"].
    """
    os.makedirs(output_dir, exist_ok=True)

    for r in metrics_rows:
        segments = r.get("traffic_segments", [])
        if not segments:
            print(f"[INFO] Sin segmentos de tráfico para {r.get('origen')} -> {r.get('destino')}, se omite mapa.")
            continue

        # Centro del mapa: primer punto del primer segmento
        first_seg = segments[0]
        coords = first_seg.get("coords", [])
        if not coords:
            continue
        center_lat, center_lon = coords[0]

        origen = r.get("origen", "origen")
        destino = r.get("destino", "destino")

        # Limpiar nombres para archivo
        def slugify(name):
            return str(name).replace(" ", "_").replace("/", "-")

        filename = f"mapa_trafico_{slugify(origen)}_{slugify(destino)}.html"
        save_path = os.path.join(output_dir, filename)

        build_traffic_map_from_segments(
            segments,
            center_lat=center_lat,
            center_lon=center_lon,
            save_path=save_path
        )

        print(f"[OK] Mapa de tráfico guardado: {save_path}")


try:
    generar_mapas_trafico(metrics_rows)
except NameError:
    print("metrics_rows no está definido en este contexto. Asegúrate de llamarlo después de evaluate_pairs_parallel.")
